<a href="https://colab.research.google.com/github/sereenajoshy/AI-ML-Intership/blob/main/DAY%207/DAY_7_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import kagglehub

path = kagglehub.dataset_download("jp797498e/twitter-entity-sentiment-analysis")

print("Path to dataset files:",path)

Using Colab cache for faster access to the 'twitter-entity-sentiment-analysis' dataset.
Path to dataset files: /kaggle/input/twitter-entity-sentiment-analysis


In [5]:
import pandas as pd

df_training = pd.read_csv(
    f'{path}/twitter_training.csv',
    header=None,
    names=['ID', 'Entity', 'Sentiment', 'Tweet']
)

df_training.head()

,ID,Entity,Sentiment,Tweet
0,2401,Borderlands,Positive,im getting on borderlands and i will murder yo...
1,2401,Borderlands,Positive,I am coming to the borders and I will kill you...
2,2401,Borderlands,Positive,im getting on borderlands and i will kill you ...
3,2401,Borderlands,Positive,im coming on borderlands and i will murder you...
4,2401,Borderlands,Positive,im getting on borderlands 2 and i will murder ...


In [6]:
df_training.describe

<bound method NDFrame.describe of          ID       Entity Sentiment  \
0      2401  Borderlands  Positive   
1      2401  Borderlands  Positive   
2      2401  Borderlands  Positive   
3      2401  Borderlands  Positive   
4      2401  Borderlands  Positive   
...     ...          ...       ...   
74677  9200       Nvidia  Positive   
74678  9200       Nvidia  Positive   
74679  9200       Nvidia  Positive   
74680  9200       Nvidia  Positive   
74681  9200       Nvidia  Positive   

                                                   Tweet  
0      im getting on borderlands and i will murder yo...  
1      I am coming to the borders and I will kill you...  
2      im getting on borderlands and i will kill you ...  
3      im coming on borderlands and i will murder you...  
4      im getting on borderlands 2 and i will murder ...  
...                                                  ...  
74677  Just realized that the Windows partition of my...  
74678  Just realized that my Mac window partition is ...  
74679  Just realized the windows partition of my Mac ...  
74680  Just realized between the windows partition of...  
74681  Just like the windows partition of my Mac is l...  

[74682 rows x 4 columns]>

In [7]:
df_training['Sentiment'].value_counts()

,count
Sentiment,
Negative,22542
Positive,20832
Neutral,18318
Irrelevant,12990


In [8]:
df = df_training[['Sentiment', 'Tweet']]
df.head()

,Sentiment,Tweet
0,Positive,im getting on borderlands and i will murder yo...
1,Positive,I am coming to the borders and I will kill you...
2,Positive,im getting on borderlands and i will kill you ...
3,Positive,im coming on borderlands and i will murder you...
4,Positive,im getting on borderlands 2 and i will murder ...


In [9]:
df = df.dropna()

print(df.isnull().sum())

Sentiment    0
Tweet        0
dtype: int64


In [11]:
df = df[df['Sentiment'] != 'Sentiment']
print(df['Sentiment'].value_counts())

Sentiment
Negative      22358
Positive      20655
Neutral       18108
Irrelevant    12875
Name: count, dtype: int64


In [12]:
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()

df['Sentiment'] = encoder.fit_transform(df['Sentiment'])

print(encoder.classes_)

['Irrelevant' 'Negative' 'Neutral' 'Positive']


In [13]:
from sklearn.model_selection import train_test_split

X = df['Tweet']
y = df['Sentiment']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(X_train.shape)
print(X_test.shape)

(59196,)
(14800,)


In [14]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

MAX_WORDS = 10000
MAX_LEN = 50

tokenizer = Tokenizer(num_words=MAX_WORDS)

tokenizer.fit_on_texts(X_train)

In [15]:
X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

In [16]:
X_train_pad = pad_sequences(
    X_train_seq,
    maxlen=MAX_LEN,
    padding='post'
)

X_test_pad = pad_sequences(
    X_test_seq,
    maxlen=MAX_LEN,
    padding='post'
)

print(X_train_pad.shape)
print(X_test_pad.shape)

(59196, 50)
(14800, 50)


In [17]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

model = Sequential([
    Embedding(input_dim=MAX_WORDS,
              output_dim=128,
              input_length=MAX_LEN),

    LSTM(64),

    Dense(32, activation='relu'),

    Dense(4, activation='softmax')
])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [18]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [19]:
history = model.fit(
    X_train_pad,
    y_train,
    epochs=5,
    batch_size=64,
    validation_split=0.2
)

Epoch 1/5
740/740 ━━━━━━━━━━━━━━━━━━━━ 63s 81ms/step - accuracy: 0.3133 - loss: 1.3627 - val_accuracy: 0.2681 - val_loss: 1.3653
Epoch 2/5
740/740 ━━━━━━━━━━━━━━━━━━━━ 60s 81ms/step - accuracy: 0.3426 - loss: 1.3472 - val_accuracy: 0.3709 - val_loss: 1.3285
Epoch 3/5
740/740 ━━━━━━━━━━━━━━━━━━━━ 59s 80ms/step - accuracy: 0.4459 - loss: 1.1960 - val_accuracy: 0.4798 - val_loss: 1.1320
Epoch 4/5
740/740 ━━━━━━━━━━━━━━━━━━━━ 60s 81ms/step - accuracy: 0.6133 - loss: 0.9183 - val_accuracy: 0.7196 - val_loss: 0.7404
Epoch 5/5
740/740 ━━━━━━━━━━━━━━━━━━━━ 58s 78ms/step - accuracy: 0.8035 - loss: 0.5382 - val_accuracy: 0.7831 - val_loss: 0.5873


In [20]:
loss, accuracy = model.evaluate(
    X_test_pad,
    y_test
)

print("Test Accuracy:", accuracy)

463/463 ━━━━━━━━━━━━━━━━━━━━ 5s 10ms/step - accuracy: 0.7793 - loss: 0.6118
Test Accuracy: 0.7793243527412415


In [21]:
predictions = model.predict(X_test_pad)

predicted_labels = predictions.argmax(axis=1)

463/463 ━━━━━━━━━━━━━━━━━━━━ 7s 15ms/step


In [22]:
from sklearn.metrics import classification_report

print(classification_report(
    y_test,
    predicted_labels,
    target_names=encoder.classes_
))

              precision    recall  f1-score   support

  Irrelevant       0.79      0.65      0.71      2575
    Negative       0.80      0.85      0.83      4472
     Neutral       0.77      0.75      0.76      3622
    Positive       0.76      0.81      0.78      4131

    accuracy                           0.78     14800
   macro avg       0.78      0.76      0.77     14800
weighted avg       0.78      0.78      0.78     14800



In [23]:
sentence = ["This game is absolutely amazing"]

seq = tokenizer.texts_to_sequences(sentence)

pad = pad_sequences(
    seq,
    maxlen=MAX_LEN,
    padding='post'
)

prediction = model.predict(pad)

print(
    encoder.inverse_transform(
        [prediction.argmax()]
    )
)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step
['Positive']


In [24]:
sentence = ["Hii"]

seq = tokenizer.texts_to_sequences(sentence)

pad = pad_sequences(
    seq,
    maxlen=MAX_LEN,
    padding='post'
)

prediction = model.predict(pad)

print(
    encoder.inverse_transform(
        [prediction.argmax()]
    )
)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
['Negative']
